### Sequences, iterables, generators: revisited

In simple terms, a container is iterable, if we can go through all its elements using a for loop. All the sequences are iterable, but there are other iterable objects as well. We can even create iterable types ourselves. In our class there needs to be a special method __iter__ that returns an iterator for the container. An iterator is an object that has method __next__, which returns the next element from the container. Let's have a look at a simple example where the container and its iterator are the same class.

In [2]:
class WeekdayIterator(object):
    """Iterator for weekdays."""

    def __init__(self):
        self.i=0  #start from monday
        self.weekdays = ("Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday")

    def __iter__(self): #If the object were a container, then this method would return the iterator over the elements of the container.
        return self

    def __next__(self):
        if self.i == 7:
            raise StopIteration #Signal that all weekdays were already iterated over
        else:
            weekdays = self.weekdays[self.i]
            self.i += 1
            return weekdays


for w in WeekdayIterator():
    print(w)


Monday
Tuesday
Wednesday
Thursday
Friday
Saturday
Sunday


## Tracing `for w in WeekdayIterator(): print(w)` — Step by Step

Let's connect this directly to your earlier `sum()`/generator internals lesson — because a `for` loop does **exactly** the same thing you traced there: repeatedly calling `__next__()` until `StopIteration`.

---

### Step 0 — Creating the Object

```python
WeekdayIterator()
```

Runs `__init__`:
```python
self.i = 0
self.weekdays = ("Monday", "Tuesday", "Wednesday", "Thursday", "Friday")
```

So the new object starts with `i=0` and holds the 5-day tuple.

---

### Step 1 — `for` Asks for an Iterator via `__iter__`

```python
for w in WeekdayIterator():
```

Behind the scenes, `for` first calls:

```python
it = WeekdayIterator().__iter__()
```

```python
def __iter__(self):
    return self          # "I AM my own iterator"
```

**`__iter__` just returns `self`** — meaning the same object serves **both** roles: it's the container AND the iterator. (This is what the code comment means: *"if the object were a container, `__iter__` would return the iterator"* — here, the object skips that separation and just hands back itself.)

---

### Step 2 — `for` Repeatedly Calls `__next__()`

This is the **exact same mechanism** from your `sum()` trace — `for` keeps calling `next()` until it gets `StopIteration`:

```python
def __next__(self):
    if self.i == 7:
        raise StopIteration
    else:
        weekday = self.weekdays[self.i]
        self.i += 1
        return weekday
```

**Call 1:**
```
self.i == 7?  → No (i is 0)
weekday = self.weekdays[0]    → "Monday"
self.i += 1                     → i becomes 1
return "Monday"
```
```
w = "Monday"  →  print(w)  →  Monday
```

**Call 2:**
```
self.i == 7?  → No (i is 1)
weekday = self.weekdays[1]    → "Tuesday"
self.i += 1                     → i becomes 2
return "Tuesday"
```
```
Tuesday
```

**Calls 3, 4, 5** continue the same way, printing `Wednesday`, `Thursday`, `Friday` — and `self.i` becomes `5`.

---

### Call 6 — Where It Actually Breaks

```
self.i == 7?  → No (i is 5)
weekday = self.weekdays[5]     ← ✗ PROBLEM!
```

**`self.weekdays` only has 5 elements** — valid indices are `0, 1, 2, 3, 4` (Monday through Friday). `self.weekdays[5]` is **out of range**:

```python
weekdays[5]
# ✗ IndexError: tuple index out of range
```

---

### There's a Real Bug in This Code

The condition should stop the loop **once `i` reaches 5** (the length of the tuple), not `7`:

```python
def __next__(self):
    if self.i == 5:            # ← should match len(self.weekdays), not 7!
        raise StopIteration
    else:
        weekday = self.weekdays[self.i]
        self.i += 1
        return weekday
```

With `== 7`, the code tries to keep going for **2 more iterations than the data actually has** — and crashes with `IndexError` on the 6th call, **before ever reaching the `i==7` check that was meant to stop it gracefully**.

---

### The CORRECTED Trace — What Should Actually Happen

```python
def __next__(self):
    if self.i == 5:          # 5 = len(self.weekdays), the FIXED version
        raise StopIteration
    else:
        weekday = self.weekdays[self.i]
        self.i += 1
        return weekday
```

```
Call 1: i=0 → "Monday"   → i becomes 1
Call 2: i=1 → "Tuesday"    → i becomes 2
Call 3: i=2 → "Wednesday"    → i becomes 3
Call 4: i=3 → "Thursday"       → i becomes 4
Call 5: i=4 → "Friday"          → i becomes 5
Call 6: i==5 → StopIteration!    ← loop ends CLEANLY here
```

**Final printed output:**
```
Monday
Tuesday
Wednesday
Thursday
Friday
```

The `for` loop catches `StopIteration` internally (same as your `sum()` trace — `except StopIteration: break`), and simply **stops looping**, without any error reaching your code.

---

### Side-by-Side — Buggy vs Fixed

| | `if self.i == 7:` (as written) | `if self.i == 5:` (correct) |
|---|---|---|
| Calls 1–5 | prints Mon–Fri normally | prints Mon–Fri normally |
| Call 6 | `weekdays[5]` → **IndexError, crash!** | `StopIteration` → loop ends cleanly |
| Final result | program crashes | 5 weekdays printed successfully |

---

### The Full Mechanism, Connected to What You Already Know

```
for w in WeekdayIterator():
     │
     ├─ 1. calls .__iter__()  →  gets back the SAME object (since __iter__ returns self)
     │
     ├─ 2. repeatedly calls .__next__()
     │        │
     │        ├─ returns "Monday", "Tuesday", ... one at a time
     │        └─ eventually raises StopIteration → for loop catches it → stops
     │
     └─ 3. print(w) runs once per value successfully returned
```

This is **identical** in structure to how `sum()` consumed your generator earlier — same `iter()`/`next()`/`StopIteration` protocol, just implemented **manually** here inside a class, instead of automatically via a generator expression.

---

### The One-Sentence Summary

> `for w in WeekdayIterator():` calls `__iter__()` once (which returns the object itself), then repeatedly calls `__next__()` — each call returns the next weekday and advances `self.i`, until `__next__()` raises `StopIteration`, which the `for` loop catches silently to end the loop. As written, however, the code has a bug: the stopping condition checks `self.i == 7`, but the tuple only has 5 valid indices, so it actually crashes with `IndexError` on the 6th call instead of stopping gracefully — the condition should be `self.i == 5` (or `self.i == len(self.weekdays)`) to match the actual data. 🎯

What if the WeekdayIterator is not a container? What would happen. Assume we have  ```__next__``` method either. what would happen in that for loop

## If `WeekdayIterator` Had No `__iter__` or `__next__` — What Happens

Let's strip both methods away and see exactly where the `for` loop breaks.

---

### The Stripped-Down Class

```python
class WeekdayIterator(object):
    """Just a plain object — no __iter__, no __next__"""

    def __init__(self):
        self.i = 0
        self.weekdays = ("Monday", "Tuesday", "Wednesday", "Thursday", "Friday")

    # NO __iter__ method
    # NO __next__ method
```

---

### Tracing `for w in WeekdayIterator(): print(w)`

Remember the mechanism from your `sum()`/generator lesson: `for` **always** starts by asking for an iterator via `__iter__`:

```python
it = WeekdayIterator().__iter__()
```

But this class **has no `__iter__` method**. Python looks for it, doesn't find it, and immediately fails:

```
TypeError: 'WeekdayIterator' object is not iterable
```

**The loop never even starts** — not one single value gets produced, not even an attempt at `Monday`. The failure happens at the **very first step** — asking "can I get an iterator from you?" — and the answer is a flat no.

---

### Confirming This Is EXACTLY the Same Rule From Earlier

Remember your `zip()` lesson:

> *"If it works in a `for` loop, it works in `zip`."*

And from `sum()`:

> *"`sum()` doesn't 'detect' that it received a generator — it simply calls `iter()` on whatever you gave it."*

**Same exact rule applies to `for` loops themselves.** `for` isn't magic — it's just another piece of Python that relies on the iterator protocol:

```python
for x in obj:
    ...
```

is really:

```python
it = iter(obj)          # calls obj.__iter__()
while True:
    try:
        x = next(it)       # calls it.__next__()
    except StopIteration:
        break
    # loop body runs with x
```

If `obj.__iter__()` doesn't exist, the **very first line** of this hidden translation already fails — the whole loop is a non-starter.

---

### Proving It Directly

```python
class Empty:
    pass

for x in Empty():
    print(x)
```

```
TypeError: 'Empty' object is not iterable
```

Compare this to trying `iter()` manually — same error, same cause:

```python
e = Empty()
iter(e)
```
```
TypeError: 'Empty' object is not iterable
```

`for` and `iter()` hit the exact same wall for the exact same reason.

---

### What If ONLY `__iter__` Exists, But NOT `__next__`?

Worth checking this middle ground too, since it reveals **where exactly** the failure moves to:

```python
class HalfBroken(object):
    def __init__(self):
        self.i = 0
        self.weekdays = ("Monday", "Tuesday")

    def __iter__(self):
        return self          # returns itself as "the iterator"

    # NO __next__ method!
```

```python
for w in HalfBroken():
    print(w)
```

**Step 1** — `__iter__()` is called → succeeds → returns `self` (no error yet, since `__iter__` genuinely exists)

**Step 2** — `for` now tries `next(it)` on that returned object → looks for `__next__` → **not found**:

```
TypeError: 'HalfBroken' object is not an iterator
```

**Notice the DIFFERENT error message this time** — *"not an iterator"* rather than *"not iterable"*. This precisely distinguishes the two roles:

| Missing method | Error | Meaning |
|---|---|---|
| `__iter__` missing | *"object is not **iterable**"* | can't even START the loop — no way to GET an iterator |
| `__next__` missing (on the returned iterator) | *"object is not an **iterator**"* | got something back from `__iter__`, but it can't actually PRODUCE values |

---

### The Two Roles, Made Explicit

This distinction maps directly onto the paragraph you read just before this exercise:

> *"there needs to be a special method `__iter__` that returns an iterator for the container. An iterator is an object that has method `__next__`"*

Two **separate** requirements, checked at **two separate moments**:

```
for x in obj:
      │
      ├─ Step 1: obj MUST have __iter__          ← "is obj ITERABLE?"
      │              (if missing → TypeError: not iterable)
      │
      └─ Step 2: whatever __iter__ RETURNED
                  MUST have __next__                ← "is the RESULT an ITERATOR?"
                  (if missing → TypeError: not an iterator)
```

In your original `WeekdayIterator`, both roles happened to be **fulfilled by the same object** (`__iter__` returns `self`, and `self` also has `__next__`) — which is a common, valid shortcut, but it can obscure that these are genuinely **two distinct jobs** being satisfied by one class.

---

### Side-by-Side Summary

| Scenario | What breaks | Error message |
|---|---|---|
| No `__iter__`, no `__next__` | `for` can't even begin | `TypeError: 'X' object is not iterable` |
| `__iter__` exists, no `__next__` | `for` starts, then fails on first value | `TypeError: 'X' object is not an iterator` |
| Both exist correctly | works fully | (prints values normally, as traced earlier) |

---

### The One-Sentence Summary

> Without `__iter__`, the `for` loop fails **immediately**, before producing anything, with `TypeError: object is not iterable` — because `for` always starts by calling `iter(obj)`, and there's nothing to call. If `__iter__` exists but the object it returns has no `__next__`, the loop gets one step further (successfully "starts") but then fails with a **different** error, `TypeError: object is not an iterator`, the moment it tries to pull the first value — cleanly demonstrating that "iterable" (has `__iter__`) and "iterator" (has `__next__`) are two separate, individually-required roles. 🎯

What if it doesn't have container but have ```__iter__``` and ```__next__```

## The Case You're Describing Is Exactly What `WeekdayIterator` Already IS!

This is a great question because it reveals something important: **your original `WeekdayIterator` was NEVER a "container" in the strict sense** — and it worked perfectly fine anyway. Let's unpack why.

---

### What "Container" Actually Requires vs What Your Class Has

Recall the container definition from a few questions back:

> *"Container = an object whose purpose is to HOLD a collection of values"* — with things like `__contains__` (for `in`), `__getitem__`/indexing, `len()`, etc.

`WeekdayIterator` doesn't support **any** of that:

```python
w = WeekdayIterator()

len(w)               # ✗ TypeError: object of type 'WeekdayIterator' has no len()
w[0]                    # ✗ TypeError: 'WeekdayIterator' object is not subscriptable
"Monday" in w              # ✗ TypeError: argument of type 'WeekdayIterator' is not iterable... wait
```

Actually, let's check that last one carefully — `in` might work through a different path. But the key point stands: **this class never claimed to be a container.** It only ever claimed to be an **iterator**.

---

### The Actual Rule — Being "Iterable" (Usable in `for`) Does NOT Require Being a "Container"

This is the precise clarification your question is pushing toward. There are really **two, independent** things an object can be:

```
Container  =  holds/stores values (has __contains__, __len__, __getitem__, etc.)
Iterable    =  can be walked through with `for` (has __iter__ returning something with __next__)
```

**These are NOT the same requirement, and NEITHER requires the other!**

```
         ┌─────────────────────────┐
         │   Objects with __iter__  │   ← "iterable" — this is ALL `for` cares about
         │   + __next__ (directly   │
         │   or via what __iter__   │
         │   returns)                │
         └───────────┬──────────────┘
                       │
          ┌────────────┴─────────────┐
          │                            │
    ALSO happen to be           are ONLY iterable,
    containers (lists,             NOTHING else
    tuples, dicts...)          (WeekdayIterator, generators,
                                  file objects, range's iterator...)
```

`WeekdayIterator` lives in the **right-hand branch** — it's iterable, full stop, with **zero** container behavior. And that's completely fine — `for` never asked for a container. It only ever asks: *"do you have `__iter__`? does that give me something with `__next__`?"*

---

### Proving `for` Doesn't Care About Container-ness

```python
class PureIterator:
    """No __len__, no __contains__, no __getitem__ — NOT a container at all."""
    def __init__(self):
        self.n = 0
    def __iter__(self):
        return self
    def __next__(self):
        if self.n >= 3:
            raise StopIteration
        self.n += 1
        return self.n

for x in PureIterator():
    print(x)
```

```
1
2
3
```

**Works perfectly** — despite this class having **none** of the container machinery whatsoever. `for` was satisfied entirely by `__iter__` + `__next__`.

---

### You've Already Met Real Examples of This!

**Generators** — remember your earlier lessons on `(x for x in range(3))`?

```python
gen = (x*10 for x in range(3))

len(gen)        # ✗ TypeError — generators have no length!
gen[0]            # ✗ TypeError — no indexing!
5 in gen             # technically works, but by CONSUMING the generator — very different from container membership testing

for x in gen:            # ✓ works FINE — this is all it needed
    print(x)
```

A generator object is **iterable, and nothing else**. No container features at all — same category as `WeekdayIterator`.

**File objects** — remember `for line in f:`?

```python
with open("data.txt") as f:
    len(f)           # ✗ TypeError — files don't support len()!
    f[0]                # ✗ TypeError — no indexing!
    for line in f:         # ✓ works — files are iterable, not containers
        print(line)
```

**`map()` and `filter()` results** — same story:

```python
m = map(str, [1, 2, 3])
len(m)          # ✗ TypeError
for x in m:        # ✓ works
    print(x)
```

**None of these are containers.** All of them are iterable.

---

### So — Direct Answer to Your Question

> *"What if it doesn't have container but have `__iter__` and `__next__`?"*

**Nothing bad happens — this is not just allowed, it's an extremely common and completely valid design.** The `for` loop works perfectly, exactly as demonstrated in your original `WeekdayIterator` trace. "Container" and "iterable" are simply **unrelated requirements** — you only need `__iter__` + `__next__` to make `for` happy; container features (`len`, indexing, `in`) are a **completely separate, optional** set of capabilities that many iterables (generators, files, custom iterators like yours) simply don't have.

---

### Why the Original Text Mentioned "Container" At All, Then

Going back to the original passage:

> *"if we can go through all its elements using a `for` loop"* ... *"there needs to be a special method `__iter__` that returns an iterator for the container"*

The text was describing the **common case** — many iterables (lists, tuples, dicts) genuinely **are** containers too, and for those, `__iter__` typically returns a **separate** helper iterator object (not `self`), precisely so the container can be iterated **multiple times independently**, while a pure iterator (like `WeekdayIterator`, or a generator) can usually only be consumed **once**.

```python
L = [1, 2, 3]          # a CONTAINER — can make MANY independent iterators from it
it1 = iter(L)
it2 = iter(L)
next(it1)                 # → 1
next(it2)                  # → 1     ← independent! it2 wasn't affected by it1
```

```python
w = WeekdayIterator()     # NOT a container — iter(w) always returns the SAME object (self)
it1 = iter(w)
it2 = iter(w)
print(it1 is it2)             # → True!  Same object — NOT independent
next(it1)                       # advances self.i
next(it2)                        # continues from where it1 left off — SHARED state!
```

This is the real practical consequence of "container that's also iterable" vs "pure iterator, not a container": a real container lets you iterate it **repeatedly, independently, from the start each time**; a pure iterator (like yours) gets **consumed once** and can't restart.

---

### The One-Sentence Summary

> Being "iterable" (usable in `for`) only requires `__iter__` + `__next__` — it has **nothing to do** with being a container. Your `WeekdayIterator` proves this: it's a perfectly valid, fully-working iterable with zero container features (no `len()`, no indexing, no `__contains__`). Generators, file objects, and `map()`/`filter()` results are all real-world examples of exactly this same category — iterable, but not containers. The practical difference only shows up if you try to iterate **twice**: a true container (like a list) gives you a fresh, independent iterator each time you call `iter()` on it, while a pure iterator like yours shares its single internal state across every `iter()` call, since `__iter__` just returns `self`. 🎯

In [3]:
from collections import abc

containers = ["efg", [1,2,3], (4,5), WeekdayIterator]

for c in containers:
    if isinstance(c,abc.Sequence):
        print(c, "is a sequence")
    else:
        print(c, "is not a sequence")

efg is a sequence
[1, 2, 3] is a sequence
(4, 5) is a sequence
<class '__main__.WeekdayIterator'> is not a sequence


Weekday is not a sequence because, for instance, you cannot index it with the brackets ```[]```, but it is an iterable:

In [4]:
isinstance(WeekdayIterator(), abc.Iterable)

True

So it is possible to create iterators ourselves, but the syntax was quite complicated. There is an easier option using generators. A generator is a function that contains a ```yield``` statement. Note the difference between generators and generator expressions we saw in the first week. Both however produce iterables. Here's an example of a generator:

In [5]:
def mydate(day=1, month=1):
    lenghts=(31,28,31,30,31,30,31,31,30,31,30,31)   # How many dates in month
    first_day = day

    for m in range (month, 13):
        for d in range(first_day, lenghts[m-1] + 1):
            yield (d,m)
        first_day = 1

# Create the generator by calling the function
gen = mydate(26, 2)    # Start from 26th of function
for i, (day, month) in enumerate(gen):
    if i == 5: break   # Print only the first five dates from the generator
    print(f"Index {i}, day {day}, month {month}")


Index 0, day 26, month 2
Index 1, day 27, month 2
Index 2, day 28, month 2
Index 3, day 29, month 2
Index 4, day 30, month 2


## Generator Functions — The Easy Way to Build an Iterator

This is showing you a **shortcut** for everything you just did manually with `WeekdayIterator`. Instead of writing a whole class with `__iter__`, `__next__`, and manual state tracking (`self.i`), Python lets you just write a **function** with a magic keyword: `yield`.

---

### The Core Idea — `yield` Pauses, Doesn't Return

```python
def simple_gen():
    yield "a"
    yield "b"
    yield "c"
```

Calling this function does **NOT** run the code immediately:

```python
g = simple_gen()
print(g)          # → <generator object simple_gen at 0x...>   ← just a generator OBJECT, nothing ran yet!
```

**This is exactly your generator-expression lazy behavior from way back**, just written with a full function body instead of `(x for x in ...)`. Calling `simple_gen()` creates a generator, but the function body hasn't executed a single line yet.

```python
next(g)     # → "a"    ← NOW it runs, until it hits the first yield, then PAUSES there
next(g)      # → "b"    ← resumes EXACTLY where it left off, runs to the next yield
next(g)       # → "c"
next(g)        # → StopIteration!   (function reached its end)
```

**`yield` is like a bookmark** — the function runs up to `yield`, hands out that value, and **freezes in place**, remembering exactly where it was (including all its local variables!). The next `next()` call **resumes** right after that `yield`, as if nothing happened.

---

### Why This Replaces Your `WeekdayIterator` Class Entirely

Compare:

**The hard way (what you just built manually):**
```python
class WeekdayIterator:
    def __init__(self):
        self.i = 0
        self.weekdays = (...)
    def __iter__(self):
        return self
    def __next__(self):
        if self.i == 5:
            raise StopIteration
        w = self.weekdays[self.i]
        self.i += 1
        return w
```

**The easy way (a generator function):**
```python
def weekday_generator():
    for day in ("Monday", "Tuesday", "Wednesday", "Thursday", "Friday"):
        yield day
```

**Both behave identically when used in a `for` loop:**
```python
for w in weekday_generator():
    print(w)
```

The generator function **automatically** gets `__iter__` and `__next__` behavior for free — Python builds all that machinery for you behind `yield`, so you never have to manually track `self.i` or manually `raise StopIteration`.

---

### Now Let's Trace YOUR `mydate` Example

```python
def mydate(day=1, month=1):
    lengths = (31,28,31,30,31,30,31,31,30,31,30,31)
    first_day = day
    for m in range(month, 13):
        for d in range(first_day, lengths[m-1] + 1):
            yield (d, m)
        first_day = 1
```

**Step 1 — Create the generator:**
```python
gen = mydate(26, 2)     # day=26, month=2 (February)
```

Nothing runs yet — just like before, this only creates the generator object.

**Step 2 — The outer loop:** `for i, (day, month) in enumerate(gen):`

This asks `gen` for values one at a time, via `next()`, exactly like your manual iterator. Let's trace what happens **inside** `mydate` each time it's asked for the next value.

---

### Tracing the Body — What `yield` Does Here

```python
first_day = day = 26        # starting point
for m in range(2, 13):        # months: Feb(2), Mar(3), ..., Dec(12)
    for d in range(first_day, lengths[m-1] + 1):
        yield (d, m)
    first_day = 1               # after the FIRST month, start future months from day 1
```

**First pass — `m = 2` (February):**
```
lengths[1] = 28    (February has 28 days)
d loops: range(26, 29)  → 26, 27, 28

yield (26, 2)    ← call 1
yield (27, 2)     ← call 2
yield (28, 2)      ← call 3
```

**Then `first_day = 1`** — so from here on, every future month starts counting days from **1**, not 26 (makes sense — 26 was only special because that's where YOU asked it to start).

**Second pass — `m = 3` (March):**
```
lengths[2] = 31    (March has 31 days)
d loops: range(1, 32)  → 1, 2, 3, ... 31

yield (1, 3)    ← call 4
yield (2, 3)     ← call 5
yield (3, 3)      ← call 6  ... etc.
```

---

### Now — The Outer Loop That Actually Prints

```python
for i, (day, month) in enumerate(gen):
    if i == 5:
        break
    print(f"Index {i}, day {day}, month {month}")
```

`enumerate(gen)` pairs each generator value with an index `0, 1, 2, ...` — same `enumerate` you've used many times. Each value from `gen` is a `(day, month)` tuple, immediately **unpacked** into `day, month` variables (tuple unpacking, from your earlier lessons).

**Tracing the first 5 iterations:**

```
i=0: next(gen) → (26, 2)  → day=26, month=2 → print "Index 0, day 26, month 2"
i=1: next(gen) → (27, 2)   → day=27, month=2 → print "Index 1, day 27, month 2"
i=2: next(gen) → (28, 2)    → day=28, month=2 → print "Index 2, day 28, month 2"
i=3: next(gen) → (1, 3)       → day=1, month=3  → print "Index 3, day 1, month 3"
i=4: next(gen) → (2, 3)        → day=2, month=3  → print "Index 4, day 2, month 3"
i=5: next(gen) → (3, 3)         → i==5 → break!  (nothing printed this time)
```

**Final output:**
```
Index 0, day 26, month 2
Index 1, day 27, month 2
Index 2, day 28, month 2
Index 3, day 1, month 3
Index 4, day 2, month 3
```

---

### The Key Insight — `break` Stops EARLY, and That's Totally Fine

Notice: `mydate` could theoretically produce **hundreds** of dates (all the way to December 31st) — but the loop only asked for **6** values before `break`ing. The generator **never computed** the rest — same lazy, memory-light behavior as `range()` and every generator expression you've studied:

```python
gen = mydate(26, 2)     # instant — nothing computed
# ... only when next() is actually called does ANY date get calculated
```

If `mydate` were a plain function that built a **full list** of all dates first, it would waste time and memory computing hundreds of dates you'll never even look at. `yield` avoids that entirely — dates are computed **one at a time, exactly when asked**, and computation simply **stops** the moment nobody asks for more.

---

### Why "Note the Difference Between Generators and Generator Expressions"

Quick distinction worth having crisp:

```python
gen_expr = (x**2 for x in range(5))     # GENERATOR EXPRESSION — one-line, simple transforms only

def gen_func():                            # GENERATOR FUNCTION — full function body,
    yield 1                                  # can have loops, conditionals, multiple yields,
    yield 2                                    # arbitrarily complex logic
```

Both **produce a generator object** and both work identically in a `for` loop — but generator **expressions** are for simple, one-shot transformations (like your `sum(len(line.split()) for line in lines)`), while generator **functions** (using `def` + `yield`) can contain **arbitrarily complex logic** — loops within loops, conditions, running state — exactly like `mydate`'s nested month/day loops, which would be impossible to squeeze into a single-line generator expression.

---

### The One-Sentence Summary

> A generator function is a regular `def` function containing `yield` — calling it creates a lazy generator object (nothing runs yet); each `next()` call resumes execution from exactly where it last paused, runs until the next `yield`, hands back that value, and pauses again — automatically implementing the full `__iter__`/`__next__` machinery you built by hand in `WeekdayIterator`, without you writing a class or tracking state manually. In `mydate`, each `yield (d, m)` produces one date; the outer loop only pulled 6 values before `break`, so only those 6 dates were ever actually computed — the rest of the year was never touched, thanks to the same lazy evaluation you've seen throughout this whole course. 🎯

Note that it would not be possible to write the above iterable using a generator expression, and it would have been very clumsy to explicitly write it as an iterator like we did the ```WeekdayIterator```.

The below figure shows the relationships between different iterables we have seen:

<img src="images/iterables.png" width="1000"/>